# Analyst Workbench

Este cuaderno te deja escoger un `ticker`, una `fecha`, un `agente` y una `peticion`.

El flujo hace esto:
- te muestra que data requiere el agente elegido
- corre automaticamente los prerequisitos necesarios
- ejecuta la etapa del agente con los modelos del repo
- genera una respuesta final orientada a tu peticion
- guarda artefactos en disco para que puedas inspeccionarlos

Importante:
- No expone chain-of-thought privado del modelo.
- Si falta data suficiente, la respuesta debe indicarlo explicitamente.

In [4]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint
from IPython.display import Markdown, display

REPO_ROOT = Path(r'C:\Users\alemu\OneDrive\Documentos\TradingAgents')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analyst_access.workbench import describe_agent, list_agents, run_agent_request

TICKER = 'MDT'
TRADE_DATE = '2026-05-18'
AGENT_KEY = 'research_manager'
USER_REQUEST = 'Revisa si hay algún elemento en sus fundamentales que se haya degradado en los últimos 6 meses y que pueda afectar el precio de la acción en los próximos 3 a 12 meses.'

# Opcional: usar los defaults del repo dejando None.
QUICK_MODEL = 'gpt-5.4-mini'
DEEP_MODEL = 'gpt-5.5'
REASONING_EFFORT = 'medium'

OUTPUT_ROOT = 'analyst_access_sessions'


In [2]:
import os

key = os.getenv("OPENAI_API_KEY")
print(key[:8], "...", key[-4:] if key else "NO KEY")

sk-proj- ... bEQA


In [3]:
import os, sys
from pathlib import Path

print("python:", sys.executable)
print("cwd:", Path.cwd())
print(".env:", Path(".env").resolve(), Path(".env").exists())

key = os.getenv("OPENAI_API_KEY")
print("key preview:", key[:8], "...", key[-4:] if key else "NO KEY")

python: c:\Users\alemu\.conda\envs\tau-research-trading-agents\python.exe
cwd: C:\Users\alemu\OneDrive\Documentos\TradingAgents
.env: C:\Users\alemu\OneDrive\Documentos\TradingAgents\.env False
key preview: sk-proj- ... bEQA


In [5]:
display(Markdown('## Agentes disponibles'))
for item in list_agents():
    print(f"- {item['agent_key']}: {item['display_name']}")
    print('  prereqs:', item['prerequisites'])
    for needed in item['required_data']:
        print('   *', needed)
    print()

## Agentes disponibles

- market_analyst: Market Analyst
  prereqs: []
   * OHLCV price history for the ticker
   * technical indicators such as moving averages, MACD, RSI, Bollinger bands, ATR, and VWMA
   * enough recent market data to support a directional market view

- social_media_analyst: Social Media Analyst
  prereqs: []
   * recent company-specific news and public discussion proxies
   * enough recent articles to infer sentiment and narrative pressure

- news_analyst: News Analyst
  prereqs: []
   * company-specific recent news
   * global and macro market news over the recent lookback window
   * enough macro context to explain possible impact on the ticker

- fundamentals_analyst: Fundamentals Analyst
  prereqs: []
   * company profile and high-level fundamentals
   * income statement data
   * balance sheet data
   * cash flow statement data

- bull_researcher: Bull Researcher
  prereqs: ['market_analyst', 'social_media_analyst', 'news_analyst', 'fundamentals_analyst']
   * market report
   * sen

In [6]:
spec = describe_agent(AGENT_KEY)
display(Markdown(f"## Agente seleccionado: `{AGENT_KEY}`"))
pprint(spec)

## Agente seleccionado: `research_manager`

{'agent_key': 'research_manager',
 'context_fields': ['investment_debate_state', 'investment_plan'],
 'display_name': 'Research Manager',
 'goal_hint': 'Synthesize the debate into a clear investment plan.',
 'kind': 'derived',
 'llm': 'deep',
 'prerequisites': ['market_analyst',
                   'social_media_analyst',
                   'news_analyst',
                   'fundamentals_analyst',
                   'bull_researcher',
                   'bear_researcher'],
 'required_data': ['bull and bear debate history',
                   'analyst reports from market, sentiment, news, and '
                   'fundamentals'],
 'stage_slug': '07_research_manager'}


In [7]:
result = run_agent_request(
    ticker=TICKER,
    trade_date=TRADE_DATE,
    agent_key=AGENT_KEY,
    user_request=USER_REQUEST,
    output_root=OUTPUT_ROOT,
    quick_model=QUICK_MODEL,
    deep_model=DEEP_MODEL,
    reasoning_effort=REASONING_EFFORT,
)

display(Markdown('## Resultado de la sesion'))
pprint({k: v for k, v in result.items() if k != 'answer'})
display(Markdown('## Respuesta del agente'))
display(Markdown(result['answer']))

## Resultado de la sesion

{'agent_key': 'research_manager',
 'display_name': 'Research Manager',
 'execution_plan': ['market_analyst',
                    'social_media_analyst',
                    'news_analyst',
                    'fundamentals_analyst',
                    'bull_researcher',
                    'bear_researcher',
                    'research_manager'],
 'output_dir': 'analyst_access_sessions\\MDT\\2026-05-18\\research_manager\\20260518_104323',
 'required_data': ['bull and bear debate history',
                   'analyst reports from market, sentiment, news, and '
                   'fundamentals'],
 'ticker': 'MDT',
 'trade_date': '2026-05-18',
 'user_request': 'Revisa si hay algún elemento en sus fundamentales que se '
                 'haya degradado en los últimos 6 meses y que pueda afectar el '
                 'precio de la acción en los próximos 3 a 12 meses.'}


## Respuesta del agente

## Conclusión rápida

Sí. Con la evidencia disponible, **sí hay deterioro fundamental relevante en MDT durante el periodo reciente**, principalmente en **márgenes, utilidad neta, EPS y calidad/visibilidad del flujo de caja**. No parece una degradación de solvencia —el balance sigue sólido—, pero sí una degradación de **rentabilidad y apalancamiento operativo**, que puede presionar la acción en los próximos **3 a 12 meses**.

Mi lectura: **MDT sigue siendo una buena franquicia médica, pero el fundamental que más importa para el precio de la acción —crecimiento de ganancias y expansión/estabilidad de márgenes— se ha deteriorado.** Por eso mantendría una postura de **vender/reducir o evitar nuevas compras** hasta ver estabilización clara.

---

## Qué se ha degradado

| Área fundamental | Evidencia disponible | Lectura |
|---|---:|---|
| Margen bruto | Bajó de aprox. **65.8%** a **63.8%** | Deterioro material de ~200 pb; indica presión de costos, mezcla o ejecución |
| Margen operativo | Bajó de aprox. **18.9%** a **17.8%** | Menor eficiencia operativa; menos apalancamiento sobre ingresos |
| Utilidad neta | Cayó de **$1.374B** a **$1.143B** | Caída significativa pese a ingresos ligeramente mayores |
| EPS | Bajó de **$1.07** a **$0.89** | Deterioro directo de la tesis de valuación |
| Ingreso operativo | Bajó de **$1.696B** a **$1.602B** | La empresa está generando menos beneficio operativo |
| Flujo de caja libre | Rebotó a **$2.3B**, pero venía de **$457M** y **$584M** | Hay recuperación, pero todavía luce volátil |
| Cargos especiales/restructuración | Aumentaron según el debate | Puede ayudar a futuro, pero en el corto plazo presiona resultados |

El punto más importante es este: **los ingresos crecieron ligeramente, pero las ganancias cayeron.**

Los ingresos pasaron de **$8.961B** a **$9.017B**, lo que muestra que la demanda no está colapsando. Pero el problema es que MDT no está convirtiendo ese crecimiento en mayor beneficio. Para una compañía madura de medtech, eso es negativo porque el mercado suele valorar la estabilidad de márgenes y la previsibilidad de EPS.

---

## Qué no se ha degradado de forma preocupante

No todo está empeorando. Hay elementos defensivos que siguen intactos:

- **Balance sólido**
  - Current ratio: **2.535**
  - Caja e inversiones de corto plazo: **$8.383B**
  - Deuda manejable
  - Deuda neta ligeramente mejorada, según el debate

- **Dividend yield atractivo**
  - Aproximadamente **3.7%**
  - Parece cubierto con la información disponible

- **Ingresos todavía crecen**
  - Revenue: **$9.017B** vs **$8.961B**

- **Franquicia diversificada**
  - Cardiovascular
  - Neurociencia
  - Medical surgical
  - Diabetes
  - Robótica, navegación quirúrgica e IA

Esto reduce el riesgo de una crisis financiera, pero no elimina el riesgo de que la acción siga débil si las ganancias no se recuperan.

---

## Por qué esto puede afectar el precio en los próximos 3 a 12 meses

El riesgo principal no es que MDT “se rompa”. El riesgo es que se mantenga como una acción barata pero sin catalizador.

### 1. Presión sobre el múltiplo

MDT cotiza alrededor de **12.7x P/E forward**, que ópticamente parece barato. Pero si el EPS sigue cayendo o si las estimaciones futuras son demasiado optimistas, el mercado puede seguir aplicando un múltiplo bajo.

Una acción puede parecer barata y aun así no subir si los inversores no confían en la trayectoria de márgenes.

### 2. Riesgo de revisiones negativas de estimaciones

La caída de EPS de **$1.07** a **$0.89** es relevante. Si los analistas empiezan a recortar expectativas de EPS para los próximos trimestres, eso puede presionar la acción incluso si los ingresos siguen creciendo modestamente.

### 3. Menor confianza en la ejecución

El mercado está pidiendo evidencia de que la reestructuración y las iniciativas de innovación realmente se traduzcan en margen y EPS. Por ahora, la evidencia visible va en la dirección contraria: menor margen bruto, menor margen operativo y menor utilidad neta.

### 4. Técnica débil que refuerza el deterioro fundamental

La acción cayó aproximadamente **26%**, de cerca de **$103.7** a **$76.15**, y está por debajo de la media de 50 días y de 200 días. Además, los indicadores MACD y RSI son débiles según la evidencia del debate.

Eso sugiere que el mercado ya está penalizando la pérdida de confianza en la trayectoria de ganancias.

---

## Mi lectura de inversión

La degradación fundamental más importante es clara: **MDT está creciendo poco en ingresos, pero está perdiendo rentabilidad.**

Eso suele ser una mala combinación para una compañía madura. El balance y el dividendo ayudan a limitar el riesgo extremo, pero no son catalizadores suficientes para que la acción se reratee en el corto plazo.

### Plan recomendado

- **Si ya se posee MDT:** reducir o vender parcialmente/materialmente la posición.
- **Si no se posee:** no iniciaría una posición larga todavía.
- **Si hay rebotes hacia resistencia técnica o medias móviles:** usaría esos rebotes para reducir exposición.
- **Para reconsiderar una postura más constructiva:** esperaría evidencia de:
  - estabilización o mejora del margen bruto;
  - mejora del margen operativo;
  - recuperación de EPS;
  - flujo de caja libre fuerte por más de un trimestre;
  - guía mantenida o elevada;
  - recuperación técnica por encima de medias de 50 y 200 días.

---

## Veredicto

Sí, **hay deterioro fundamental relevante** en MDT en los datos recientes. La degradación está concentrada en **márgenes, EPS, utilidad neta y visibilidad de flujo de caja**, no en solvencia o liquidez.

Para los próximos **3 a 12 meses**, esto puede afectar negativamente el precio porque el mercado probablemente exigirá pruebas de recuperación antes de volver a pagar un múltiplo más alto. Hasta que MDT demuestre estabilización de márgenes y recuperación de EPS, la acción parece más un caso de **“buena compañía, mala acción”**.

**Postura: vender/reducir o evitar nuevas compras.**

---

## MISSING DATA:

1. Reportes completos de analistas de mercado, sentimiento, noticias y fundamentales mencionados como requeridos pero no incluidos en el bundle.  
2. Estados financieros trimestrales completos de MDT de los últimos dos trimestres para confirmar con precisión la degradación de los últimos 6 meses.  
3. Evolución de guidance de ingresos, margen operativo y EPS durante los últimos 6 meses.  
4. Desglose por segmento de crecimiento, margen y mix para identificar qué unidades explican la compresión de rentabilidad.

In [ ]:
session_dir = Path(result['output_dir'])
display(Markdown(f'## Carpeta de salida\n`{session_dir}`'))

for path in sorted(session_dir.iterdir()):
    print(path.name)

In [ ]:
summary_path = session_dir / 'session_summary.json'
if summary_path.exists():
    display(Markdown('## Session summary'))
    pprint(json.loads(summary_path.read_text(encoding='utf-8')))

answer_path = session_dir / 'selected_answer.md'
if answer_path.exists():
    display(Markdown('## Respuesta guardada en disco'))
    display(Markdown(answer_path.read_text(encoding='utf-8')))

In [ ]:
display(Markdown('## Etapas ejecutadas y archivos generados'))
for path in sorted([p for p in session_dir.iterdir() if p.is_dir()]):
    print(f'[{path.name}]')
    for child in sorted(path.iterdir()):
        print('  -', child.name)
    print()